<a href="https://colab.research.google.com/github/traceopt-ai/traceml/blob/main/notebooks/huggingface_trl_lora_gradient_accumulation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Same effective batch, different microbatch: a TRL LoRA diagnostic

Fine-tune **Qwen3-1.7B** with **TRL + LoRA** and answer one practical question:

> If the effective batch stays at 4, does using a larger physical microbatch make each optimizer update faster—and how much GPU memory does it cost?

This is a controlled diagnostic, not a model-quality benchmark. The two default runs keep the model, data, sequence length, optimizer-step count, seed, and effective batch fixed. Only the physical batch and gradient-accumulation count change.

**Runtime:** Google Colab with a T4 GPU. The default two-run experiment is the recommended path; a third memory-boundary case is optional.


## Why this comparison is valid

For a single GPU:

```
effective batch = per-device batch × gradient accumulation
```

The default lanes are therefore equivalent at the optimizer-update level. Every example is also padded or truncated to 512 tokens, so both lanes process the same token count:

| Run | Physical batch | Accumulation | Effective batch | Purpose |
|---|---:|---:|---:|---|
| A | 1 | 4 | 4 | memory-saving baseline |
| B | 2 | 2 | 4 | primary comparison |
| C (optional) | 4 | 1 | 4 | find the T4 memory boundary |

TraceML brackets one Hugging Face **optimizer update** as one step. When accumulation is enabled, the forward and backward calls from all microbatches are added inside that step. This makes optimizer-step timing comparable across the lanes.

Accelerate may prepare ordinary input tensors before that callback window, so host-to-device timing can be unavailable here. We will use Hugging Face Trainer runtime for the end-to-end result and TraceML for phase timing, GPU utilization, and peak memory.


## 1. Configure the experiment

Thirty optimizer updates are enough for a quick diagnostic. Keep the two required runs enabled; turn on the third only if you want to test the largest physical batch.


In [ ]:
MAX_STEPS = 30
DATASET_SAMPLES = 256
RUN_THIRD_CASE = False

## 2. Check the GPU

In Colab, choose **Runtime → Change runtime type → T4 GPU** before running this cell.


In [ ]:
import subprocess
import torch

subprocess.run(["nvidia-smi"], check=False)
assert torch.cuda.is_available(), "Enable a T4 GPU in the Colab runtime first."
print("GPU:", torch.cuda.get_device_name(0))

## 3. Install the small dependency set

The workload uses ordinary FP16 LoRA—no quantization library and no Hugging Face login are required.


In [ ]:
%pip install -q -U "traceml-ai[hf]" "trl[peft]" datasets

## 4. The workload

Each run uses:

- `Qwen/Qwen3-1.7B`
- `trl-lib/Capybara`
- FP16 LoRA with rank 16
- every example padded or truncated to 512 tokens
- gradient checkpointing enabled in every lane
- the same shuffled subset and seed

The runs execute in separate processes through `traceml run`. That keeps model state, CUDA peak counters, and instrumentation isolated.


### TraceML integration

The training code remains a normal `SFTTrainer`. The integration is two lines:

```python
traceml_hf.init()
callbacks=[traceml_hf.TraceMLTrainerCallback()]
```

The complete script is shown below so the notebook is self-contained.


In [ ]:
%%writefile trl_lora_gradient_accumulation.py
"""Compare LoRA microbatch layouts with one fixed effective batch size.

Designed for the companion Colab notebook. Each invocation runs one isolated
TRL ``SFTTrainer`` configuration so model state, CUDA peak counters, and
TraceML instrumentation cannot leak between comparison lanes.
"""

from __future__ import annotations

import argparse
import json
from pathlib import Path

import torch
from datasets import load_dataset
from peft import LoraConfig
from transformers import set_seed
from trl import SFTConfig, SFTTrainer

from traceml_ai.integrations import huggingface as traceml_hf


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser()
    parser.add_argument("--model-id", default="Qwen/Qwen3-1.7B")
    parser.add_argument("--dataset-id", default="trl-lib/Capybara")
    parser.add_argument("--dataset-samples", type=int, default=256)
    parser.add_argument("--max-length", type=int, default=512)
    parser.add_argument("--max-steps", type=int, default=30)
    parser.add_argument("--per-device-batch-size", type=int, required=True)
    parser.add_argument(
        "--gradient-accumulation-steps", type=int, required=True
    )
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--output-root", default="outputs")
    return parser.parse_args()


def main() -> None:
    args = parse_args()
    if not torch.cuda.is_available():
        raise RuntimeError(
            "This experiment requires a CUDA GPU. In Colab, select "
            "Runtime -> Change runtime type -> T4 GPU."
        )

    set_seed(args.seed)
    effective_batch = (
        args.per_device_batch_size * args.gradient_accumulation_steps
    )
    run_name = (
        f"bs{args.per_device_batch_size}_"
        f"ga{args.gradient_accumulation_steps}"
    )
    output_dir = Path(args.output_root) / run_name
    output_dir.mkdir(parents=True, exist_ok=True)

    print(
        f"[experiment] run={run_name} gpu={torch.cuda.get_device_name(0)!r} "
        f"physical_batch={args.per_device_batch_size} "
        f"gradient_accumulation={args.gradient_accumulation_steps} "
        f"effective_batch={effective_batch} max_length={args.max_length} "
        f"optimizer_steps={args.max_steps}",
        flush=True,
    )

    dataset = load_dataset(args.dataset_id, split="train")
    sample_count = min(args.dataset_samples, len(dataset))
    dataset = dataset.shuffle(seed=args.seed).select(range(sample_count))

    peft_config = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.0,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules="all-linear",
    )

    training_args = SFTConfig(
        output_dir=str(output_dir),
        model_init_kwargs={"dtype": "float16", "use_cache": False},
        per_device_train_batch_size=args.per_device_batch_size,
        gradient_accumulation_steps=args.gradient_accumulation_steps,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        max_length=args.max_length,
        # With max_length truncation, this pads every sequence to exactly the
        # same length and controls token work across microbatch layouts.
        pad_to_multiple_of=args.max_length,
        packing=False,
        max_steps=args.max_steps,
        warmup_steps=5,
        learning_rate=2e-4,
        optim="adamw_torch",
        logging_steps=5,
        logging_first_step=True,
        save_strategy="no",
        report_to="none",
        disable_tqdm=True,
        fp16=True,
        bf16=False,
        dataloader_num_workers=0,
        dataset_num_proc=1,
        seed=args.seed,
        data_seed=args.seed,
    )

    # TraceML records one step per optimizer update. All forward/backward
    # calls from the accumulation group are summed into that step.
    traceml_hf.init()
    trainer = SFTTrainer(
        model=args.model_id,
        args=training_args,
        train_dataset=dataset,
        peft_config=peft_config,
        callbacks=[traceml_hf.TraceMLTrainerCallback()],
    )

    train_output = trainer.train()
    metrics = dict(train_output.metrics)
    metrics.update(
        {
            "run_name": run_name,
            "physical_batch_size": args.per_device_batch_size,
            "gradient_accumulation_steps": (
                args.gradient_accumulation_steps
            ),
            "effective_batch_size": effective_batch,
            "max_length": args.max_length,
            "optimizer_steps": args.max_steps,
            "gpu_name": torch.cuda.get_device_name(0),
        }
    )
    metrics_path = output_dir / "trainer_metrics.json"
    metrics_path.write_text(
        json.dumps(metrics, indent=2, sort_keys=True), encoding="utf-8"
    )
    print(f"[experiment] Trainer metrics: {metrics_path}", flush=True)


if __name__ == "__main__":
    main()


## 5. Run the two required lanes

Run A uses four microbatches per optimizer update. Run B uses two larger microbatches. Both consume four examples per update and perform the same number of optimizer updates.

The first model download is cached, so the second run starts faster.


In [ ]:
!traceml run --mode summary --logs-dir logs --run-name bs1_ga4 trl_lora_gradient_accumulation.py --args --per-device-batch-size 1 --gradient-accumulation-steps 4 --max-steps {MAX_STEPS} --dataset-samples {DATASET_SAMPLES}

In [ ]:
!traceml run --mode summary --logs-dir logs --run-name bs2_ga2 trl_lora_gradient_accumulation.py --args --per-device-batch-size 2 --gradient-accumulation-steps 2 --max-steps {MAX_STEPS} --dataset-samples {DATASET_SAMPLES}

## 6. Compare the saved TraceML summaries

The command prints the main changes and writes a reusable JSON comparison.


In [ ]:
!traceml compare logs/bs1_ga4/final_summary.json logs/bs2_ga2/final_summary.json --output=logs/bs1_ga4_vs_bs2_ga2

### Optional: test physical batch 4

This lane removes accumulation while preserving the effective batch. It is useful as a memory-boundary test, but it is not required for the story. If it runs out of memory on your T4, that is itself the result: physical batch 2 is the practical upper lane for this setup.


In [ ]:
if RUN_THIRD_CASE:
    !traceml run --mode summary --logs-dir logs --run-name bs4_ga1 trl_lora_gradient_accumulation.py --args --per-device-batch-size 4 --gradient-accumulation-steps 1 --max-steps {MAX_STEPS} --dataset-samples {DATASET_SAMPLES}
    !traceml compare logs/bs1_ga4/final_summary.json logs/bs4_ga1/final_summary.json --output=logs/bs1_ga4_vs_bs4_ga1
else:
    print("Skipped. Set RUN_THIRD_CASE = True to test physical batch 4.")

## 7. Put the decision metrics in one table

Trainer runtime answers whether the complete training call improved. TraceML explains the optimizer-step phases and memory trade-off. Values shown as `None` were not available in that run; the raw compare output above remains the source of truth.


In [ ]:
import json
from pathlib import Path

import pandas as pd


def read_json(path):
    with Path(path).open(encoding="utf-8") as handle:
        return json.load(handle)


def metric(compare_payload, section, name, side):
    return (
        compare_payload.get("sections", {})
        .get(section, {})
        .get("metrics", {})
        .get(name, {})
        .get(side)
    )


def rounded(value, digits=2, scale=1.0):
    if value is None:
        return None
    return round(float(value) / scale, digits)


primary = read_json("logs/bs1_ga4_vs_bs2_ga2.json")
run_specs = [
    ("bs1_ga4", 1, 4, primary, "lhs"),
    ("bs2_ga2", 2, 2, primary, "rhs"),
]

if RUN_THIRD_CASE and Path("logs/bs1_ga4_vs_bs4_ga1.json").exists():
    optional = read_json("logs/bs1_ga4_vs_bs4_ga1.json")
    run_specs.append(("bs4_ga1", 4, 1, optional, "rhs"))

rows = []
for run_name, physical_batch, accumulation, comparison, side in run_specs:
    trainer_metrics = read_json(f"outputs/{run_name}/trainer_metrics.json")
    rows.append(
        {
            "run": run_name,
            "physical batch": physical_batch,
            "accumulation": accumulation,
            "effective batch": physical_batch * accumulation,
            "Trainer runtime (s)": rounded(
                trainer_metrics.get("train_runtime")
            ),
            "Trainer steps/s": rounded(
                trainer_metrics.get("train_steps_per_second"), 3
            ),
            "TraceML step (ms)": rounded(
                metric(comparison, "step_time", "step_time_ms", side)
            ),
            "forward total (ms)": rounded(
                metric(comparison, "step_time", "forward_ms", side)
            ),
            "backward total (ms)": rounded(
                metric(comparison, "step_time", "backward_ms", side)
            ),
            "optimizer (ms)": rounded(
                metric(comparison, "step_time", "optimizer_ms", side)
            ),
            "peak reserved (GiB)": rounded(
                metric(
                    comparison,
                    "step_memory",
                    "peak_reserved_bytes",
                    side,
                ),
                scale=1024**3,
            ),
            "GPU util (%)": rounded(
                metric(
                    comparison,
                    "system",
                    "gpu_util_avg_percent",
                    side,
                )
            ),
        }
    )

results = pd.DataFrame(rows)
results

## How to read the result

- **Lower Trainer runtime and higher Trainer steps/s** mean the complete training job improved.
- **Lower TraceML optimizer-step time with higher peak memory** means the larger physical batch traded memory for throughput.
- **Similar timings** mean batch 1 already kept this model/GPU busy enough; do not manufacture a bottleneck claim.
- **An out-of-memory third lane** identifies a real capacity boundary, not a failed experiment.
- Do not interpret missing H2D timing as zero transfer time; Accelerate can move inputs before the traced optimizer-step window.

One pass is enough to learn the workflow. Before publishing exact percentages, restart the runtime and repeat each lane three times, then report the median.

## Apply the pattern to your own trainer

Keep the effective batch, model, data order, sequence length, precision, checkpointing, and optimizer-step count fixed. Change only the physical batch and accumulation count. That turns a toy speed claim into a controlled hardware-specific decision.
